In [1]:
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer
from datasets import load_dataset

# Import custom scripts
from data_utils import ContiguousDataLoader
from trainer import train_universal_model
from evals_utils import plot_training_metrics

# models
from models.FLB_Model import FLB_Transformer 

# Optimize PyTorch for your hardware
torch.backends.cudnn.benchmark = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


## Hyperparameter Configuration

In [2]:
# We use GPT-2's vocabulary size since we are using its tokenizer
VOCAB_SIZE = 50257  

# Model Dimensions
HIDDEN_DIM = 256
NUM_LAYERS = 6
NUM_HEADS = 8

# TBPTT & Sequence Length
SEQ_LEN = 128
BATCH_SIZE = 16 
ACCUMULATION_STEPS = 4 # Effective batch size = 16 * 4 = 64

# Training Parameters
LEARNING_RATE = 3e-4
EPOCHS = 1
LOG_INTERVAL = 50

## Data Ingestion (WikiText-2)

In [3]:
print("Loading WikiText-2 and GPT-2 Tokenizer...")
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')

# Filter out empty lines and join the text into one massive string
text_data = "\n".join([text for text in dataset['text'] if text.strip() != ""])

# Tokenize the entire corpus into a single 1D array
print("Tokenizing corpus (this might take a minute)...")
tokens = tokenizer.encode(text_data, return_tensors='pt').squeeze(0)
print(f"Total tokens in dataset: {len(tokens):,}")

# Initialize your custom contiguous data loader
dataloader = ContiguousDataLoader(tokens, batch_size=BATCH_SIZE, seq_len=SEQ_LEN)
print(f"Created {len(dataloader)} micro-batches per epoch.")

Loading WikiText-2 and GPT-2 Tokenizer...


/home/abrah/miniconda3/envs/flb-transformer/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Tokenizing corpus (this might take a minute)...


Token indices sequence length is longer than the specified maximum sequence length for this model (2391884 > 1024). Running this sequence through the model will result in indexing errors


Total tokens in dataset: 2,391,884
Created 1167 micro-batches per epoch.


## Model Instantiation

In [4]:
model = FLB_Transformer(
    vocab_size=VOCAB_SIZE, 
    hidden_dim=HIDDEN_DIM, 
    num_layers=NUM_LAYERS, 
    seq_len=SEQ_LEN, 
    batch_size=BATCH_SIZE, # Note: Pass the MICRO-batch size to the engine!
    dropout=0.1, 
    aux_weight=0.05
).to(device)

# Count parameters to ensure fair benchmarking later
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FLB-Transformer initialized with {total_params:,} trainable parameters.")

FLB-Transformer initialized with 28,938,321 trainable parameters.


## Training Execution

In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print("Starting training...")
# This will safely log to training_log.csv without bottlenecking your GPU
trained_model = train_universal_model(
    model=model, 
    dataloader=dataloader, 
    optimizer=optimizer, 
    epochs=EPOCHS, 
    accumulation_steps=ACCUMULATION_STEPS, 
    log_interval=LOG_INTERVAL,
    device=device,
    log_file='artifacts/logs/flb_training_log.csv',
    save_dir='artifacts/checkpoints'
)

Starting training...


Epoch 1/1:   0%|          | 0/1167 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Artifact Generation

In [ ]:
print("Generating thesis artifacts...")
plot_training_metrics(log_file='flb_training_log.csv', output_dir='flb_plots')

# Display one of the plots right here in the notebook just to check it
from IPython.display import Image, display
display(Image(filename='flb_plots/loss_curve.png'))